In [ ]:
import pandas as pd
import numpy as np

weather = pd.read_csv("../data/weather_archive.csv", parse_dates=["date"])
locations = pd.read_csv("../data/locations.csv")

def to_week_start(dates):
    return dates - pd.to_timedelta((dates.dt.dayofweek + 1) % 2, unit="D")

weather["week_start"] = to_week_start(weather["date"])

weekly = weather.groupby(["location_id", "week_start"]).agg(
    rainfall_mm=("rainfall_mm", "sum"),
    n_days=("date", "count"),
).reset_index()

weekly = weekly[weekly["n_days"] == 2]
weekly = weekly.merge(locations[["id", "name"]], left_on="location_id", right_on="id")

rain = weekly.pivot(index="week_start", columns="name", values="rainfall_mm")
print(rain.shape)
rain.tail()

(0, 0)


name
week_start


In [2]:
def naive(series, t):
    return series.iloc[t - 1]

def seasonal_naive(series, t):
    return series.iloc[t - 52]

def seasonal_mean(series, t):
    vals = []
    k = 1
    while t - 52 * k - 1 >= 0:
        vals.extend(series.iloc[t - 52 * k - 1 : t - 52 * k + 2].tolist())
        k += 1
    return float(np.mean(vals))

def walk_forward(series, forecaster, n_test=156):
    start = len(series) - n_test
    preds, actuals = [], []
    for t in range(start, len(series)):
        preds.append(forecaster(series, t))
        actuals.append(series.iloc[t])
    return np.array(actuals), np.array(preds)

def mae(a, p):
    return np.mean(np.abs(a - p))

def rmse(a, p):
    return np.sqrt(np.mean((a - p) ** 2))

In [3]:
rows = []
for name in rain.columns:
    s = rain[name].dropna().reset_index(drop=True)
    for label, f in [("naive", naive), ("seasonal_naive", seasonal_naive), ("seasonal_mean", seasonal_mean)]:
        a, p = walk_forward(s, f)
        rows.append({"destination": name, "method": label, "MAE": mae(a, p), "RMSE": rmse(a, p)})

results = pd.DataFrame(rows)
print("MAE (mm per week)")
display(results.pivot(index="destination", columns="method", values="MAE").round(1))
print("RMSE (mm per week)")
display(results.pivot(index="destination", columns="method", values="RMSE").round(1))

MAE (mm per week)


method,naive,seasonal_mean,seasonal_naive
destination,,,
Alleppey,38.6,32.7,48.7
Athirappilly Falls,58.1,48.5,67.9
Cherrapunji,55.7,55.8,49.1
India Gate,15.3,12.9,19.0
Jaisalmer,5.2,5.4,7.3
Munnar,56.7,45.1,62.5
Mysore Palace,15.0,13.3,17.1
Taj Mahal,15.1,12.2,18.2


RMSE (mm per week)


method,naive,seasonal_mean,seasonal_naive
destination,,,
Alleppey,59.6,51.1,73.4
Athirappilly Falls,96.7,73.9,107.4
Cherrapunji,110.9,87.1,96.4
India Gate,32.2,26.1,36.2
Jaisalmer,12.0,10.5,15.8
Munnar,97.4,72.1,103.3
Mysore Palace,26.5,19.5,26.3
Taj Mahal,31.9,25.3,35.6


In [4]:
def seasonal_median(series, t):
    vals = []
    k = 1
    while t - 52 * k - 1 >= 0:
        vals.extend(series.iloc[t - 52 * k - 1 : t - 52 * k + 2].tolist())
        k += 1
    return float(np.median(vals))

rows = []
for name in rain.columns:
    s = rain[name].dropna().reset_index(drop=True)
    a, p = walk_forward(s, seasonal_median)
    rows.append({"destination": name, "method": "seasonal_median", "MAE": mae(a, p), "RMSE": rmse(a, p)})

results = pd.concat([results, pd.DataFrame(rows)], ignore_index=True)
print("MAE (mm per week)")
display(results.pivot(index="destination", columns="method", values="MAE").round(1))

MAE (mm per week)


method,naive,seasonal_mean,seasonal_median,seasonal_naive
destination,,,,
Alleppey,38.6,32.7,33.5,48.7
Athirappilly Falls,58.1,48.5,45.3,67.9
Cherrapunji,55.7,55.8,43.5,49.1
India Gate,15.3,12.9,12.5,19.0
Jaisalmer,5.2,5.4,4.1,7.3
Munnar,56.7,45.1,43.2,62.5
Mysore Palace,15.0,13.3,11.5,17.1
Taj Mahal,15.1,12.2,11.3,18.2


In [6]:
import xgboost as xgb

def make_features(series, t):
    week = series.index[t].isocalendar().week
    return [
        series.iloc[t - 1],
        series.iloc[t - 2],
        series.iloc[t - 3],
        series.iloc[t - 4],
        series.iloc[t - 52],
        seasonal_mean(series, t),
        np.sin(2 * np.pi * week / 52),
        np.cos(2 * np.pi * week / 52),
    ]

In [8]:
def xgb_walk_forward(series, n_test=156, retrain_every=13):
    offset = 53
    X_all = np.array([make_features(series, t) for t in range(offset, len(series))])
    y_all = series.iloc[offset:].to_numpy()

    start = len(series) - n_test
    preds = []
    for block_start in range(start, len(series), retrain_every):
        block_end = min(block_start + retrain_every, len(series))
        model = xgb.XGBRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=42,
        )
        model.fit(X_all[: block_start - offset], np.log1p(y_all[: block_start - offset]))
        block = X_all[block_start - offset : block_end - offset]
        preds.extend(np.expm1(model.predict(block)).clip(min=0))

    actuals = series.iloc[start:].to_numpy()
    return actuals, np.array(preds)

In [9]:
rows = []
for name in rain.columns:
    s = rain[name].dropna()
    a, p = xgb_walk_forward(s)
    rows.append({"destination": name, "method": "xgboost", "MAE": mae(a, p), "RMSE": rmse(a, p)})

all_results = pd.concat([results, pd.DataFrame(rows)])
table = all_results.pivot(index="destination", columns="method", values="MAE").round(1)
table["gain_vs_seasonal_mean_%"] = (
    (table["seasonal_mean"] - table["xgboost"]) / table["seasonal_mean"] * 100
).round(1)
table

method,naive,seasonal_mean,seasonal_median,seasonal_naive,xgboost,gain_vs_seasonal_mean_%
destination,,,,,,
Alleppey,38.6,32.7,33.5,48.7,32.7,0.0
Athirappilly Falls,58.1,48.5,45.3,67.9,43.1,11.1
Cherrapunji,55.7,55.8,43.5,49.1,43.7,21.7
India Gate,15.3,12.9,12.5,19.0,12.8,0.8
Jaisalmer,5.2,5.4,4.1,7.3,4.1,24.1
Munnar,56.7,45.1,43.2,62.5,45.6,-1.1
Mysore Palace,15.0,13.3,11.5,17.1,12.0,9.8
Taj Mahal,15.1,12.2,11.3,18.2,13.6,-11.5


In [10]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

def fourier(index, K=3):
    week = index.isocalendar().week.to_numpy().astype(float)
    cols = []
    for k in range(1, K + 1):
        cols.append(np.sin(2 * np.pi * k * week / 52))
        cols.append(np.cos(2 * np.pi * k * week / 52))
    return np.column_stack(cols)

def sarimax_walk_forward(series, n_test=156, retrain_every=13):
    y = np.log1p(series.to_numpy())
    X = fourier(series.index)
    start = len(series) - n_test
    preds = []
    for b in range(start, len(series), retrain_every):
        e = min(b + retrain_every, len(series))
        res = SARIMAX(y[:b], exog=X[:b], order=(1, 0, 1)).fit(disp=False)
        ext = res.append(y[b:e], exog=X[b:e], refit=False)
        pm = ext.get_prediction(start=b, end=e - 1, dynamic=False).predicted_mean
        preds.extend(np.expm1(pm).clip(min=0))
    return series.iloc[start:].to_numpy(), np.array(preds)

In [11]:
rows = []
for name in rain.columns:
    s = rain[name].dropna()
    a, p = sarimax_walk_forward(s)
    rows.append({"destination": name, "method": "sarimax_fourier", "MAE": mae(a, p), "RMSE": rmse(a, p)})

all_results = pd.concat([all_results, pd.DataFrame(rows)], ignore_index=True)
for metric in ["MAE", "RMSE"]:
    print(metric, "(mm per week)")
    display(all_results.pivot(index="destination", columns="method", values=metric).round(1))

MAE (mm per week)


method,naive,sarimax_fourier,seasonal_mean,seasonal_median,seasonal_naive,xgboost
destination,,,,,,
Alleppey,38.6,32.1,32.7,33.5,48.7,32.7
Athirappilly Falls,58.1,43.0,48.5,45.3,67.9,43.1
Cherrapunji,55.7,40.9,55.8,43.5,49.1,43.7
India Gate,15.3,12.0,12.9,12.5,19.0,12.8
Jaisalmer,5.2,4.3,5.4,4.1,7.3,4.1
Munnar,56.7,42.1,45.1,43.2,62.5,45.6
Mysore Palace,15.0,11.9,13.3,11.5,17.1,12.0
Taj Mahal,15.1,11.9,12.2,11.3,18.2,13.6


RMSE (mm per week)


method,naive,sarimax_fourier,seasonal_mean,seasonal_median,seasonal_naive,xgboost
destination,,,,,,
Alleppey,59.6,50.5,51.1,55.1,73.4,56.8
Athirappilly Falls,96.7,75.3,73.9,73.4,107.4,74.7
Cherrapunji,110.9,84.1,87.1,81.5,96.4,83.5
India Gate,32.2,27.9,26.1,28.1,36.2,27.5
Jaisalmer,12.0,11.4,10.5,11.7,15.8,10.9
Munnar,97.4,75.2,72.1,75.5,103.3,74.3
Mysore Palace,26.5,21.7,19.5,19.3,26.3,20.3
Taj Mahal,31.9,28.2,25.3,26.7,35.6,30.3


In [12]:
def skill_table(res, metric="RMSE"):
    t = res.pivot(index="destination", columns="method", values=metric)
    baselines = ["naive", "seasonal_naive", "seasonal_mean", "seasonal_median"]
    best = t[baselines].min(axis=1)
    return pd.DataFrame({
        "best_baseline": t[baselines].idxmin(axis=1),
        "best_baseline_error": best.round(1),
        "xgboost_vs_best_%": ((best - t["xgboost"]) / best * 100).round(1),
        "sarimax_vs_best_%": ((best - t["sarimax_fourier"]) / best * 100).round(1),
    })

print("RAINFALL, RMSE")
display(skill_table(all_results, "RMSE"))
print("RAINFALL, MAE")
display(skill_table(all_results, "MAE"))

RAINFALL, RMSE


,best_baseline,best_baseline_error,xgboost_vs_best_%,sarimax_vs_best_%
destination,,,,
Alleppey,seasonal_mean,51.1,-11.2,1.3
Athirappilly Falls,seasonal_median,73.4,-1.7,-2.5
Cherrapunji,seasonal_median,81.5,-2.4,-3.2
India Gate,seasonal_mean,26.1,-5.2,-6.8
Jaisalmer,seasonal_mean,10.5,-3.2,-8.1
Munnar,seasonal_mean,72.1,-3.0,-4.2
Mysore Palace,seasonal_median,19.3,-5.1,-12.1
Taj Mahal,seasonal_mean,25.3,-19.7,-11.4


RAINFALL, MAE


,best_baseline,best_baseline_error,xgboost_vs_best_%,sarimax_vs_best_%
destination,,,,
Alleppey,seasonal_mean,32.7,0.1,1.9
Athirappilly Falls,seasonal_median,45.3,4.7,5.1
Cherrapunji,seasonal_median,43.5,-0.5,5.9
India Gate,seasonal_median,12.5,-2.0,3.9
Jaisalmer,seasonal_median,4.1,0.1,-2.9
Munnar,seasonal_median,43.2,-5.5,2.5
Mysore Palace,seasonal_median,11.5,-4.7,-3.3
Taj Mahal,seasonal_median,11.3,-19.7,-5.3


In [13]:
weekly_t = weather.groupby(["location_id", "week_start"]).agg(
    temp=("temp_mean", "mean"), n_days=("date", "count")
).reset_index()
weekly_t = weekly_t[weekly_t["n_days"] == 7].merge(
    locations[["id", "name"]], left_on="location_id", right_on="id"
)
temp = weekly_t.pivot(index="week_start", columns="name", values="temp")

def xgb_wf(series, n_test=156, retrain_every=13, log=False):
    fwd = np.log1p if log else (lambda v: v)
    inv = np.expm1 if log else (lambda v: v)
    offset = 53
    X_all = np.array([make_features(series, t) for t in range(offset, len(series))])
    y_all = series.iloc[offset:].to_numpy()
    start = len(series) - n_test
    preds = []
    for b in range(start, len(series), retrain_every):
        e = min(b + retrain_every, len(series))
        m = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                             subsample=0.8, random_state=42)
        m.fit(X_all[: b - offset], fwd(y_all[: b - offset]))
        preds.extend(inv(m.predict(X_all[b - offset : e - offset])))
    preds = np.array(preds)
    return series.iloc[start:].to_numpy(), (preds.clip(min=0) if log else preds)

def sarimax_wf(series, n_test=156, retrain_every=13, log=False):
    y = np.log1p(series.to_numpy()) if log else series.to_numpy()
    X = fourier(series.index)
    start = len(series) - n_test
    preds = []
    for b in range(start, len(series), retrain_every):
        e = min(b + retrain_every, len(series))
        res = SARIMAX(y[:b], exog=X[:b], order=(1, 0, 1)).fit(disp=False)
        ext = res.append(y[b:e], exog=X[b:e], refit=False)
        pm = ext.get_prediction(start=b, end=e - 1, dynamic=False).predicted_mean
        preds.extend(np.expm1(pm).clip(min=0) if log else pm)
    return series.iloc[start:].to_numpy(), np.array(preds)

In [14]:
rows = []
for name in temp.columns:
    s = temp[name].dropna()
    s0 = s.reset_index(drop=True)
    for label, f in [("naive", naive), ("seasonal_naive", seasonal_naive),
                     ("seasonal_mean", seasonal_mean), ("seasonal_median", seasonal_median)]:
        a, p = walk_forward(s0, f)
        rows.append({"destination": name, "method": label, "MAE": mae(a, p), "RMSE": rmse(a, p)})
    a, p = xgb_wf(s)
    rows.append({"destination": name, "method": "xgboost", "MAE": mae(a, p), "RMSE": rmse(a, p)})
    a, p = sarimax_wf(s)
    rows.append({"destination": name, "method": "sarimax_fourier", "MAE": mae(a, p), "RMSE": rmse(a, p)})

temp_results = pd.DataFrame(rows)
print("TEMPERATURE, RMSE (°C)")
display(temp_results.pivot(index="destination", columns="method", values="RMSE").round(2))
print("TEMPERATURE, skill vs best baseline")
display(skill_table(temp_results, "RMSE"))

TEMPERATURE, RMSE (°C)


method,naive,sarimax_fourier,seasonal_mean,seasonal_median,seasonal_naive,xgboost
destination,,,,,,
Alleppey,0.57,0.52,0.77,0.78,0.91,0.54
Athirappilly Falls,0.74,0.68,0.81,0.82,1.21,0.72
Cherrapunji,1.11,0.90,1.00,1.03,1.29,0.91
India Gate,2.30,1.88,2.06,2.07,2.69,1.88
Jaisalmer,2.13,1.82,2.09,2.15,2.90,1.77
Munnar,0.56,0.49,0.77,0.72,1.02,0.51
Mysore Palace,0.80,0.68,1.01,1.03,1.56,0.80
Taj Mahal,2.14,1.70,1.81,1.81,2.46,1.71


TEMPERATURE, skill vs best baseline


,best_baseline,best_baseline_error,xgboost_vs_best_%,sarimax_vs_best_%
destination,,,,
Alleppey,naive,0.6,4.1,7.6
Athirappilly Falls,naive,0.7,3.4,8.4
Cherrapunji,seasonal_mean,1.0,9.6,10.8
India Gate,seasonal_mean,2.1,8.8,8.7
Jaisalmer,seasonal_mean,2.1,15.7,12.9
Munnar,naive,0.6,9.0,13.8
Mysore Palace,naive,0.8,-0.3,14.6
Taj Mahal,seasonal_mean,1.8,5.5,6.2
